# owner-parent string formatting code

NOTE: owner/parent research is incomplete; presume all owners are parents if we haven't updated them yet

This script will only be successful if all 'parents' identified in sheet (2/3) are included in the parent metadata on sheet (3/3)

In [1]:
import pandas
import pygsheets
import datetime
import numpy

In [2]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')

In [3]:
#ownership_tracker_data = spreadsheet.worksheet('title', 'Ownership tracker handoff - 2024/11/09').get_as_df()
ownership_tracker_data = spreadsheet.worksheet('title', 'Ownership tracker handoff - 2025/02/28').get_as_df()
ownership_tracker_data.head()

,source,Pipeline Tracker Name,id,name + LET,name_local,nameOther,gemParents,Parent1,Parent1%,Parent2,...,Parent11,Parent11%,Parent12,Parent12%,Parent13,Parent13%,Parent14,Parent14%,Parent15,Parent15%
0,Pipelines,"3Bear Energy, LLC",E100001013982,3Bear Energy LLC,,[],unknown,unknown,,,...,,,,,,,,,,
1,Pipelines,A/S Norske Shell,E100001013986,Norske Shell A/S,,['A/S Norske Shell'],Shell PLC [100.0%],Shell PLC,100.00%,,...,,,,,,,,,,
2,Pipelines,AB Amber Grid,E100001013992,Amber Grid AB,,['AB Amber Grid'],Amber Grid AB [100%],Amber Grid AB,100.00%,,...,,,,,,,,,,
3,Pipelines,Abu Dhabi Crude Oil Pipeline LLC,E100001014027,Abu Dhabi Crude Oil Pipeline LLC,,[],Abu Dhabi National Oil Co [100.0%],Abu Dhabi National Oil Co,100.00%,,...,,,,,,,,,,
4,Pipelines,Abu Dhabi National Energy Company,E100000001078,Abu Dhabi National Energy Company PJSC,شركة أبوظبى الوطنية للطاقة - ش م ع,[],Abu Dhabi National Energy Company PJSC [100%],Abu Dhabi National Energy Company PJSC,100.00%,,...,,,,,,,,,,


In [4]:
pipe_oo = spreadsheet.worksheet('title', 'Pipeline operators/owners').get_as_df(start='A2')
pipe_opr = spreadsheet.worksheet('title', 'Owner–parent relationships (2/3)').get_as_df(start='A3')

### removed unused owners

...cleaning up data before merging

In [5]:
owner_col_names = []
for num in range(1,11+1):
    owner_col = f'Owner{num}'
    owner_col_names.append(owner_col)

In [6]:
owner_UNIQUE = pipe_oo[owner_col_names].stack().dropna().unique().tolist()

In [7]:
ot_names = ownership_tracker_data['Pipeline Tracker Name'].tolist()
#for index,row in pipe_opr.iterrows():
#    if row['Owner'] not in ot_names:
#        print(row['Owner'])

In [8]:
set(ot_names)-set(owner_UNIQUE)

{'3Bear Energy, LLC',
 'ADIA',
 'AES Corporation',
 'AMBO Pipeline Limited',
 'ANR Pipeline Company',
 'AWE Limited',
 'Abu Dhabi National Energy Company',
 'Aiteo Eastern E&P Company Limited',
 'Alashankou Duoadau Pipeline Co., Ltd.',
 'Alashankou Horizon Oil & Gas Co., Ltd.',
 'Alaska Gasline Development Corporation',
 'Alberta Investment Management Company',
 'Algonquin Power & Utilities Corporation',
 'Amberjack Pipeline Company LLC',
 'Andhra Pradesh Gas Distribution Corporation Limited',
 'Anhui Huishang Great Wall Energy Co., Ltd.',
 'Anhui Province Natural Gas Development Co., Ltd.',
 'Apache Corporation',
 'Arab Petroleum Pipelines Company (SUMED)',
 'Aramco Gulf Operations Company Limited',
 'ArcelorMittal',
 'Assam Gas Co., Ltd.',
 'Atlantic Richfield Company',
 'Atmos Energy Corporation',
 'Azerbaijan International Operating Company (AIOC)',
 'BBL Company',
 'BCI',
 'BG Overseas Holding Limited',
 'BG Overseas Holdings Limited',
 'BH-Gas d.o.o.',
 'BOTAS',
 'BP',
 'Bahrain 

In [9]:
set(owner_UNIQUE)-set(ot_names)

{'',
 '3Bear Energy LLC',
 'AES Corp',
 'AMBO Pipeline Ltd',
 'ANR Pipeline Co',
 'APA SPP Pty Ltd',
 'ATCO Gas and Pipelines Ltd',
 'AWE Ltd',
 'Abu Dhabi National Energy Co',
 'Administración Nacional de la Seguridad Social (ANSES)',
 'Agencia de Promoción de la Inversión Privada – PROINVERSIÓN',
 'Aiteo Eastern E&P Co Ltd',
 'Alashankou Duoadau Pipeline Co Ltd',
 'Alashankou Horizon Oil & Gas Co Ltd',
 'Alaska Gasline Development Corp',
 'Alberta Investment Management Co',
 'Algonquin Power & Utilities Corp',
 'Alliance LP',
 'Alyeska Pipeline Service Co',
 'Amberjack Pipeline Co LLC',
 'Andhra Pradesh Gas Distribution Corp Ltd',
 'Anhui Huishang Great Wall Energy Co Ltd',
 'Anhui Longrun New Energy Co Ltd',
 'Anhui Province Natural Gas Development Co Ltd',
 'Anhui United Natural Gas Co Ltd',
 'Anhui Zhongyi Agricultural Hefu Natural Gas Co.Ltd',
 'Apache Corp',
 'Arab Petroleum Pipelines Co (SUMED)',
 'Aramco',
 'Aramco Gulf Operations Co Ltd',
 'Arm Energy Holdings LLC',
 'Assam G

## rubber meets road

In [10]:
#fuel_type = 'Gas'
#fuel_type = 'Oil'
fuel_type = 'Oil-and-Gas'

gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')

spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')
gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

if fuel_type == 'Gas':
    pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)
if fuel_type == 'Oil':
    pipes_df_orig = oil_pipes.copy()
if fuel_type == 'Oil-and-Gas':
    pipes_df_orig = pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

#pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Wiki!='']
pipes_df_orig.replace('--', numpy.nan, inplace=True)

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_10105/936881417.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pipes_df_orig.replace('--', numpy.nan, inplace=True)


In [11]:
owners_df_orig = pipe_oo.copy() #spreadsheet.worksheet('title', 'Pipeline operators/owners').get_as_df(start='A2')
owners_df_orig = owners_df_orig.loc[owners_df_orig.ProjectID!='']
owners_df = owners_df_orig.replace('',numpy.nan)

# owner_parent_links_df = spreadsheet.worksheet('title', 'Owner–parent relationships (2/3)').get_as_df(start='A2')
# owner_parent_links_df = owner_parent_links_df.loc[owner_parent_links_df['Parent–Owner Relationship Checked?']=='yes']

owner_parent_links_df = ownership_tracker_data.copy()
owner_parent_links_df.replace('',numpy.nan,inplace=True)

#parents_df = spreadsheet.worksheet('title', 'Parent metadata (3/3)').get_as_df(start='A2')
#parents_df = parents_df.loc[parents_df.Parent!='']

owners_df.set_index('ProjectID', inplace=True)
owner_parent_links_df.set_index('Pipeline Tracker Name', inplace=True)
#parents_df.set_index('Parent', inplace=True)

In [12]:
# ****************************************
## create list of owner and parent column names
owner_pct_col_names = []
owner_col_names = []

parent_pct_col_names = []
parent_col_names = []

for num in range(1,11+1):
    owner_pct_col = f'Owner{num}%'
    owner_pct_col_names.append(owner_pct_col)
    
    owner_col = f'Owner{num}'
    owner_col_names.append(owner_col)

for num in range(1,15+1):
    parent_pct_col = f'Parent{num}%'
    parent_pct_col_names.append(parent_pct_col)
    
    parent_col = f'Parent{num}'
    parent_col_names.append(parent_col)

# ****************************************
# # when there's a QCCOwner(业主单位) owner and NO entry in Owner 1 column, fill with QCC info
# qcc_owners_df = owners_df.loc[#(owners_df.Country=='China')&
#                               (~owners_df['QCCOwner(业主单位)'].isnull())& # where QCCOwner col has entry
#                               (owners_df['Owner1'].isnull())]#& # and where there's no Owner1 entry
#                               #(~owners_df['QCCOwner(业主单位)'].isin(owner_parent_links_df.index))]

# owners_df.loc[qcc_owners_df.index,'Owner1'] = qcc_owners_df['QCCOwner(业主单位)']
# owners_df.loc[qcc_owners_df.index,'Owner1%'] = '100.00%'

# ****************************************
# ## fill in missing parent info by borrowing owner info
# ## for example, if we don't have parent info, presume owner is parent for now...
# owners_FULL_set = owners_df[owner_col_names].stack().dropna().unique().tolist() # from owners_df
# owners_researched_set = list(set(owner_parent_links_df.index.to_list()))#+['Unknown'] # only existing owners, plus 'Unknown'
# owners_diff = list(set(owners_FULL_set)-set(owners_researched_set))
# owners_diff.append('unknown')
# #print(owners_diff)

# # update owner_parent_links_df with these extra owners
# owner_parent_links_df = pandas.concat([owner_parent_links_df, pandas.DataFrame(index=owners_diff, columns=owner_parent_links_df.columns)])
# # owner_parent_links_df['Parent1'].loc[owners_diff] = owners_diff
# # owner_parent_links_df['Parent1%'].loc[owners_diff] = '100.00%'
# owner_parent_links_df.loc[owners_diff,'Parent1'] = owners_diff
# owner_parent_links_df.loc[owners_diff,'Parent1%'] = '100.00%'

# # ****************************************
# # update parents_df with these as well
# # note countries will be unknkown...
# parents_set = list(set(parents_df.index.to_list()))
# parents_diff = list(set(owners_diff)-set(parents_set))
# #parents_diff.append('Unknown') # doesnt' seem necessary
# parents_df = pandas.concat([parents_df, pandas.DataFrame(numpy.nan, index=parents_diff, columns=parents_df.columns)])
# parents_df.replace(numpy.nan, 'unknown', inplace=True)

print missing parents

In [15]:
projectid_set = list(set(owners_df.index.to_list()))

# make dictionary to house parent owner info

In [17]:
po_dict = {}

# iterate through owners_df
# store in the big po_dict

for project_id,row in list(owners_df.iterrows()):
    print(project_id)
    po_dict[project_id] = {}
    
    owner_list_drop_nans = row[owner_col_names].dropna().tolist()
    owner_pct_vals = list(row[owner_pct_col_names].str.strip('%').astype('float').array/100.)[:owner_list_drop_nans.__len__()]

    #print(owner_list_drop_nans)
    #print(owner_pct_vals)
    # now go through the owner list, if it's empty create an uknown
    # if not empty, for each owner:
    #    save its percent ownership (make sure it's nan if it doesn't exist)
    #    save its list of parents (EVERY OWNER has a parent in the database)
    #    save the list of parent ownership (make sure it's a list of nans that is same length as list of parents)
    
    if owner_list_drop_nans==[]:
        owner='unknown'
        parent='unknown'
        owner_entity_id='unknown'
        owner_list = [owner]
        owner_entity_id_list = [owner_entity_id]
        parent_list = [parent]
        owner_pct_vals = [numpy.nan]
        parent_pct_vals = [numpy.nan]
        
        # if there are no owners/parents, make them unknown/unknown
        po_dict[project_id]['owner_parent_links'] = {}
        po_dict[project_id]['owner_list'] = owner_list
        po_dict[project_id]['owner_pct_vals'] = owner_pct_vals
        po_dict[project_id]['owner_entity_id_list'] = owner_entity_id_list
        po_dict[project_id]['owner_parent_links'][owner] = {}
        po_dict[project_id]['owner_parent_links'][owner]['owner_pct_val'] = owner_pct_vals[0] # record the specific fraction val of the owner
        
        po_dict[project_id]['owner_parent_links'][owner]['parent_list'] = parent_list
        po_dict[project_id]['owner_parent_links'][owner]['parent_pct_vals'] = parent_pct_vals
        po_dict[project_id]['owner_parent_links'][owner]['parent_hq_country_list'] = ['unknown']
    
    else:
        po_dict[project_id]['owner_parent_links'] = {}
        #print(po_dict)
        
        for o_idx,owner in enumerate(owner_list_drop_nans):
            parent_list_drop_nans = owner_parent_links_df.loc[owner][parent_col_names].squeeze().dropna().tolist()
            parent_pct_vals = list(owner_parent_links_df.loc[owner][parent_pct_col_names].str.strip('%').astype('float').array/100.)[:parent_list_drop_nans.__len__()] # only as long as parent_list

            po_dict[project_id]['owner_list'] = owner_list_drop_nans
            po_dict[project_id]['owner_pct_vals'] = owner_pct_vals
            # get owner entity ids from ownership tracker's work
            po_dict[project_id]['owner_entity_id_list'] = owner_parent_links_df.loc[owner_list_drop_nans].id.tolist()
            
            po_dict[project_id]['owner_parent_links'][owner] = {}
            po_dict[project_id]['owner_parent_links'][owner]['owner_pct_val'] = owner_pct_vals[o_idx]
            
            po_dict[project_id]['owner_parent_links'][owner]['parent_list'] = parent_list_drop_nans
            po_dict[project_id]['owner_parent_links'][owner]['parent_pct_vals'] = parent_pct_vals
            

P0001
P0002
P0004
P0005
P0006


KeyError: "['Shell plc'] not in index"

In [18]:
pid_list = list(po_dict.keys())

owner_parent_strings_df = pandas.DataFrame(index=pid_list, columns=[#'OwnerList','ParentList',
                                                                    'OwnerString','ParentString'])
#                                                                    'OwnerPercentsArrayWithNans','ParentPercentsArrayWithNans',
#                                                                    'OwnerPercentsArray','ParentPercentsArray',
#                                                                    'ParentOwnrshpArray'])

In [19]:
for project_id in pid_list:

    print(project_id)
    
    # get list of owners
    owner_list = po_dict[project_id]['owner_list']#.keys()
    owner_entity_id_list = po_dict[project_id]['owner_entity_id_list']
    print(owner_entity_id_list)
    owner_pct_vals = po_dict[project_id]['owner_pct_vals']
    all_parents_list = []
    all_parents_normalized_pct_vals = []
    #all_parents_hq_country_list = []
    
    for owner in owner_list:
        owner_pct_val = [ po_dict[project_id]['owner_parent_links'][owner]['owner_pct_val'] ]
        parent_list = po_dict[project_id]['owner_parent_links'][owner]['parent_list']
        #parent_hq_country_list = po_dict[project_id]['owner_parent_links'][owner]['parent_hq_country_list']
        
        parent_pct_vals = numpy.array(po_dict[project_id]['owner_parent_links'][owner]['parent_pct_vals'])
        
        parent_normalized_pct_vals = list(parent_pct_vals * owner_pct_val)
        
        all_parents_list += parent_list
        all_parents_normalized_pct_vals += parent_normalized_pct_vals
        #all_parents_hq_country_list += parent_hq_country_list
    
    owner_frac_df = pandas.DataFrame({'Owners':owner_list,'OwnerFractions':owner_pct_vals})#.dropna(how='all') # drop nan rows
    
    #print(all_parents_list)
    #print(all_parents_normalized_pct_vals)
    #print(all_parents_hq_country_list)
    parent_frac_df = pandas.DataFrame({'Parents':all_parents_list,
                                       'ParentFractions':all_parents_normalized_pct_vals,}
                                       #'ParentHQCountries':all_parents_hq_country_list}
                                     )
    
    # sum any of the same owners/parents
    owner_frac_df = pandas.DataFrame(owner_frac_df.groupby(by=['Owners'], dropna=False)['OwnerFractions'].sum(min_count=1))
    parent_frac_df = pandas.DataFrame(parent_frac_df.groupby(by=['Parents'], dropna=False)['ParentFractions'].sum(min_count=1))
    
    owner_frac_df.sort_values('OwnerFractions', ascending=False, inplace=True)
    parent_frac_df.sort_values('ParentFractions', ascending=False, inplace=True)
    
    #parent_hq_country_list = [parents_df.loc[p].ParentHQCountry for p in parent_frac_df.index.tolist()]

    parent_formatted_string = ('; ').join(list(parent_frac_df.index + (parent_frac_df['ParentFractions']*100).map(' [{:,.2f}%]'.format)))
    owner_formatted_string = ('; ').join(list(owner_frac_df.index + (owner_frac_df['OwnerFractions']*100).map(' [{:,.2f}%]'.format)))
    #parent_hq_country_formatted_string = ('; ').join(parent_hq_country_list)
    owner_entity_id_formatted_string = ('; ').join(owner_entity_id_list)

    parent_formatted_string = parent_formatted_string.replace('nan%', 'unknown %')
    owner_formatted_string = owner_formatted_string.replace('nan%', 'unknown %')
    
    owner_parent_strings_df.loc[project_id,'OwnerString'] = owner_formatted_string
    owner_parent_strings_df.loc[project_id,'OwnerEntityIDString'] = owner_entity_id_formatted_string
    owner_parent_strings_df.loc[project_id,'ParentString'] = parent_formatted_string
    
    owner_parent_strings_df.replace('','--',inplace=True)

P0001
['E100000000112']
P0002
['E100000000112', 'E100001014203']
P0004
['E100000000112']
P0005
['E100001015671']
P0006


KeyError: 'owner_entity_id_list'

# write out data as Excel file

In [380]:
now_string = datetime.datetime.now().strftime('%Y-%m-%d')
owner_parent_strings_df[['OwnerString','OwnerEntityIDString','ParentString']].to_excel('GEM-pipelines-owner-parent-strings-'+now_string+'.xlsx')
owner_parent_strings_df.to_excel('GEM-pipelines-owner-parent-strings-'+now_string+'.xlsx')